In [1]:
import os
import numpy as np
import cv2
import librosa
import soundfile as sf
from tqdm import tqdm

# ============ frequency_remap_hires (same as last tested, n_fft_out=8192) ============
def frequency_remap_hires(spectrogram, sr=22050, n_fft_out=4096, target_low=1000.0, target_high=4000.0):
    """
    Same log curve as frequency_remap (f_new = a*log(f_old+1)+b), source and
    destination resolution decoupled via n_fft_out. Forward push + midpoint
    ownership - every source row contributes, nothing silently skipped.
    """
    n_src, n_frames = spectrogram.shape
    n_bins_out = n_fft_out // 2 + 1

    freq_axis_src = np.linspace(0, sr / 2, n_src)

    b = target_low
    a = (target_high - target_low) / np.log(freq_axis_src[-1] + 1)

    f_new = a * np.log(freq_axis_src + 1) + b
    dst_pos = f_new / (sr / n_fft_out)

    boundaries = (dst_pos[:-1] + dst_pos[1:]) / 2.0

    remapped = np.zeros((n_bins_out, n_frames), dtype=spectrogram.dtype)
    freq_axis_out = np.linspace(0, sr / 2, n_bins_out)
    in_band = (freq_axis_out >= target_low) & (freq_axis_out <= target_high)

    for d in np.where(in_band)[0]:
        src_lo, src_hi = np.searchsorted(boundaries, [d - 0.5, d + 0.5])
        src_lo = min(src_lo, n_src - 1)
        src_hi = min(max(src_hi, src_lo + 1), n_src)
        remapped[d, :] = spectrogram[src_lo:src_hi, :].mean(axis=0)

    return remapped, a, b
# ============ END ============


def image_to_audio_dataset_hires(input_root, output_root, sr=22050, n_fft_out=8192):
    """
    Same structure as your sportsball batch script - os.walk, mirrored
    output folders, skip-existing resumability, per-file error handling,
    summary counters. The only thing swapped out is the sonification step
    itself: frequency_remap_hires (1-4kHz confined) + griffinlim sized to
    n_fft_out, instead of the plain full-spectrum griffinlim call.
    """
    # Source image resolution - the image's own detail, NOT tied to n_fft_out
    # (that decoupling is the whole point of _hires). Stays 513 regardless.
    target_height = 513
    target_width = 128

    os.makedirs(output_root, exist_ok=True)
    print(f"Output root created at: {output_root}")

    # Griffin-Lim parameters, driven by n_fft_out (same 50% overlap ratio as before)
    n_fft = n_fft_out
    hop_length = n_fft_out // 2
    win_length = n_fft_out

    total_files = 0
    processed_files = 0
    skipped_files = 0
    error_files = 0

    for root, dirs, files in os.walk(input_root):
        for file in tqdm(files, desc=f"Processing {os.path.basename(root)}"):
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                total_files += 1
                try:
                    relative_path = os.path.relpath(root, input_root)
                    output_dir = os.path.join(output_root, relative_path)
                    os.makedirs(output_dir, exist_ok=True)

                    input_path = os.path.join(root, file)
                    output_path = os.path.join(output_dir, f"{os.path.splitext(file)[0]}.wav")

                    if os.path.exists(output_path):
                        skipped_files += 1
                        continue

                    image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
                    if image is None:
                        print(f"\nWarning: Could not read image {input_path}")
                        error_files += 1
                        continue

                    image = cv2.resize(image, (target_width, target_height))
                    image_normalized = np.flipud(image.astype(np.float32) / 255.0)

                    spectrogram_db = librosa.amplitude_to_db(image_normalized + 1e-7, ref=np.max)
                    spectrogram_linear = librosa.db_to_amplitude(spectrogram_db)

                    # ============ NEW: 1-4kHz hi-res remap before Griffin-Lim ============
                    spectrogram_linear, a, b = frequency_remap_hires(spectrogram_linear, sr=sr, n_fft_out=n_fft_out)
                    # ============ END NEW ============

                    audio = librosa.griffinlim(
                        spectrogram_linear,
                        n_iter=32,
                        n_fft=n_fft,
                        hop_length=hop_length,
                        win_length=win_length
                    )

                    sf.write(output_path, audio, sr)
                    processed_files += 1

                except Exception as e:
                    print(f"\nError processing {file}: {str(e)}")
                    error_files += 1
                    continue

    print("\nConversion Summary:")
    print(f"Total image files found: {total_files}")
    print(f"Successfully processed: {processed_files}")
    print(f"Skipped (already existed): {skipped_files}")
    print(f"Errors encountered: {error_files}")

# Example usage - VERIFY THESE PATHS!
input_dataset_root = r"D:\Documents\Iquisitionis\103\vehicle-type-recognition\Dataset"
output_dataset_root = r"D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5"

if not os.path.exists(input_dataset_root):
    print(f"ERROR: Input path does not exist: {input_dataset_root}")
else:
    print(f"Input directory verified: {input_dataset_root}")
    image_to_audio_dataset_hires(input_dataset_root, output_dataset_root, n_fft_out=8192)
    print(f"\nCheck output directory: {output_dataset_root}")

Input directory verified: D:\Documents\Iquisitionis\103\vehicle-type-recognition\Dataset
Output root created at: D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5


Processing Dataset: 0it [00:00, ?it/s]
Processing Truck: 100%|██████████████████████████████████████████████████████████████| 100/100 [01:44<00:00,  1.05s/it]


Conversion Summary:
Total image files found: 400
Successfully processed: 400
Skipped (already existed): 0
Errors encountered: 0

Check output directory: D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5
